In [ ]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson

In [ ]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [ ]:
# award importance to a channel according to their importance ranking in a time point
def update_channel_points_linear(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 10-i
    return channel_point_dict

# award importance to a channel according to their importance ranking in a time point
def update_channel_points_same(top_k_channel_names, channel_point_dict):
    for i, channel_name in enumerate(top_k_channel_names):
        channel_point_dict[channel_name] += 1
    return channel_point_dict

def get_top_k_channels_per_timepoint(time_point_data, ch_names, k):
    # trial is of shape channels x timepoins
    # for each timepoint, find indices of top_k channels with highest activations 
    top_k_channels_indices = np.argsort(time_point_data)[::-1][:k]
    # get the channel names of the top_k channels 
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    return top_k_channels_names


# adapt so you can see a running list of top channes per trial
def get_top_k_channels_per_trial(trial, ch_names, channel_point_dict , k, update_func):
    for time_point in range(trial.shape[1]):
        #print(f"time_point: {time_point}")
        top_k_channel_names = get_top_k_channels_per_timepoint(trial[:,time_point], ch_names, k)
        # update the points of the top_k channels of the timepoint according to the (linear, stricly monotone falling) update function
        # 
        # NOTE while assumption that all trials are equally important holds, assumption that all timepoints are equally important is very questionable. 
        # Maybe think about this part again
        channel_point_dict = update_func(top_k_channel_names, channel_point_dict)
    return channel_point_dict

def get_top_k_channels(aggregate_dict, aggregate_dict_key, ch_names, k, update_func):
    # get the channel importance for each channel according to the explanation method
    # a ranking approach is chosen s.t. each timepoint in a trial and each trial is equally important

    # init dict for storing points per channel

    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0
    
    data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each
    for trial in data:
        #print(trial.shape)
        channel_point_dict = get_top_k_channels_per_trial(trial, ch_names, channel_point_dict, k, update_func)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)
    
    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    return top_k_channels, channel_point_dicts_over_time
    
def format_point_progression(point_progressions, ch_names):
    # this function collects the points over time per channel properly s.t. the point progression over time can easily be plotted
    point_progression_dict = {}
    for ch in ch_names:
        point_progression_dict[ch] = []


    for dict in point_progressions:
        for ch in ch_names:
            point_progression_dict[ch].append(dict[ch])

    return point_progression_dict


cwd = os.getcwd()
dir_path = os.path.join(cwd, "frequency_power_data")

freq_bands = {"theta": (4, 8),
              "alpha": (8, 12),
              "beta": (12, 30),
              "gamma": (30, 45)}






In [ ]:
def top_k_channel_per_trial_weighted(trial, ch_names,  channel_points_dict, k, update_func, aggregation_func=np.mean, **kwargs):
    # aggregate the importances award to each "pixel" in the trial over the time_point dimension according to the provided aggreagation_func
    #print(trial.shape)
    agg_points_per_channel = aggregation_func(trial, **kwargs)

    #print(agg_points_per_channel.shape)
    # get the indices of the top k channels
    top_k_channels_indices = np.argsort(agg_points_per_channel)[::-1][:k]
    top_k_channels_names = np.take(ch_names, top_k_channels_indices)
    #print(top_k_channels_names)
    # update the channels points according to the provided update func
    channel_points_dict = update_func(top_k_channels_names, channel_points_dict)
    return channel_points_dict

def get_top_k_weighted(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    #the previous top-k function considers each timepoint and each trial equally important.
    #in this version now each trial is still assumed to be equally important.
    #However, each timepoints importance is now weighted by the importance attributed to it by the attribution method.
    #we take the mean over the timepoint dimension. On one hand this increases susceptability to outliers, on the other hand it better allows
    #for some timepoints to be more important than others which I assume(for now) can absolutely be the case
    channel_point_dicts_over_time = []

    channel_point_dict = {}
    for ch in ch_names:
        channel_point_dict[ch] = 0

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()
    # go through each trial in the dataset and assign points for each



    for trial in data:
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, aggregation_func=np.mean, axis=1)
        channel_point_dicts_over_time.append(copy.deepcopy(channel_point_dict))

    top_k_channels = sorted(channel_point_dict, key=channel_point_dict.get, reverse=True)[:k]
    
    return top_k_channels, channel_point_dicts_over_time
    

def get_top_k_weighted_individual(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices=np.array([])):
    # unlike the upper version which collects cumulative points per channel over trials
    # this function outputs the top-k chanels per trial for each trial individually
    channel_point_dicts_individual_trials = []

    if groupby_indices.any():
        data = aggregate_dict[aggregate_dict_key].squeeze()[groupby_indices]
    else:
        data = aggregate_dict[aggregate_dict_key].squeeze()

    for trial in data:
        channel_point_dict = {}
        for ch in ch_names:
            channel_point_dict[ch] = 0

        #print(trial.shape)
        channel_point_dict = top_k_channel_per_trial_weighted(trial, ch_names, channel_point_dict, k, update_func, axis=1)
        channel_point_dicts_individual_trials.append(copy.deepcopy(channel_point_dict))
        #print(channel_point_dicts_over_time)

    temp = format_point_progression(channel_point_dicts_individual_trials, ch_names)
    sum_over_channel_points_dict = {key: sum(value) for key,value in temp.items()}
    top_k_channels = sorted(sum_over_channel_points_dict, key=sum_over_channel_points_dict.get, reverse=True)[:k]
    #top_k_channels_dict = {ch: sum_over_channel_points_dict[ch] for ch in top_k_channels}
    

    return top_k_channels, channel_point_dicts_individual_trials,sum_over_channel_points_dict


def top_k_groupby(aggregate_dict, aggregate_dict_key, groupby_dict, ch_names, k, update_func, top_k_func=get_top_k_weighted):
    groupby_channel_points_dict =  {}
    for groupby_key, groupby_indices in groupby_dict.items():
        groupby_channel_points_dict[groupby_key] = top_k_func(aggregate_dict, aggregate_dict_key, ch_names, k, update_func, groupby_indices)
    
    return groupby_channel_points_dict
 

In [ ]:

top_60_per_subject_final = {}
top_60_per_subject_dict_final = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    data = np.load(f"explanations_subject_{subject_index}.npy")
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    top_k_channels, channel_point_dicts_individual_trials, top_k_channels_dict = get_top_k_weighted_individual({"gradshap": np.abs(data)}, "gradshap", ch_names, 60, update_channel_points_linear)
    top_60_per_subject_final[subject_index] = top_k_channels
    top_60_per_subject_dict_final[subject_index] = top_k_channels_dict



In [ ]:
np.save("top_k_abs_final.npy",top_60_per_subject_dict_final)

In [ ]:

top_60_per_subject_final = {}
top_60_per_subject_dict_final = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    data = np.load(f"explanations_subject_{subject_index}_pretrain.npy")
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    top_k_channels, channel_point_dicts_individual_trials, top_k_channels_dict = get_top_k_weighted_individual({"gradshap": np.abs(data)}, "gradshap", ch_names, 60, update_channel_points_linear)
    top_60_per_subject_final[subject_index] = top_k_channels
    top_60_per_subject_dict_final[subject_index] = top_k_channels_dict

In [ ]:
np.save("top_k_abs_pretrain.npy",top_60_per_subject_dict_final)